# Batch Fraud Analytics Pipeline
### End-to-end AWS data engineering project

**Stack:** Python · AWS S3 · AWS Glue · Amazon Athena · Plotly

**Dataset:** 300,000 synthetic credit card transactions generated programmatically (no external downloads required)

---

| Phase | Description | Service |
|---|---|---|
| 1 | Generate dataset & upload to S3 | Colab + S3 |
| 2 | Schema discovery | AWS Glue Crawler |
| 3 | Clean & transform to Parquet | AWS Glue ETL |
| 4 | SQL fraud flagging | Amazon Athena |
| 5 | Interactive dashboard | Plotly + GitHub Pages |

**Credentials:** Loaded from Colab Secrets — never hardcoded. Add `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`, `AWS_REGION`, and `S3_SUFFIX` via the 🔑 Secrets icon in the left sidebar.

## Phase 1 — Data Generation & S3 Upload

### 1.1 Install dependencies

In [1]:
!pip install boto3 python-dotenv faker --quiet
print('✅ Dependencies installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 7.1 MB/s eta 0:00:00
✅ Dependencies installed


### 1.2 Load AWS credentials

In [2]:
import os

def load_credentials():
    # ── Colab Secrets ──────────────────────────────────────────────────────────
    try:
        from google.colab import userdata
        creds = {
            'AWS_ACCESS_KEY_ID':     userdata.get('AWS_ACCESS_KEY_ID'),
            'AWS_SECRET_ACCESS_KEY': userdata.get('AWS_SECRET_ACCESS_KEY'),
            'AWS_REGION':            userdata.get('AWS_REGION') or 'us-east-1',
            'S3_SUFFIX':             userdata.get('S3_SUFFIX'),
        }
        if creds['AWS_ACCESS_KEY_ID']:
            print('🔑 Credentials loaded from Colab Secrets')
            return creds
    except Exception:
        pass

    # ── .env file (local) ──────────────────────────────────────────────────────
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass

    creds = {
        'AWS_ACCESS_KEY_ID':     os.getenv('AWS_ACCESS_KEY_ID'),
        'AWS_SECRET_ACCESS_KEY': os.getenv('AWS_SECRET_ACCESS_KEY'),
        'AWS_REGION':            os.getenv('AWS_REGION', 'us-east-1'),
        'S3_SUFFIX':             os.getenv('S3_SUFFIX'),
    }
    if creds['AWS_ACCESS_KEY_ID']:
        print('🔑 Credentials loaded from .env file')
        return creds

    raise EnvironmentError(
        'No credentials found.\n'
        '  In Colab: add secrets via the 🔑 sidebar.\n'
        '  Locally:  create a .env file in the project root.'
    )

creds = load_credentials()

AWS_ACCESS_KEY_ID     = creds['AWS_ACCESS_KEY_ID']
AWS_SECRET_ACCESS_KEY = creds['AWS_SECRET_ACCESS_KEY']
AWS_REGION            = creds['AWS_REGION']
S3_SUFFIX             = creds['S3_SUFFIX']

RAW_BUCKET       = f'fraud-raw-{S3_SUFFIX}'
PROCESSED_BUCKET = f'fraud-processed-{S3_SUFFIX}'

missing = [k for k, v in creds.items() if not v]
if missing:
    raise EnvironmentError(f'Missing secrets: {missing}')

print(f'   Region:           {AWS_REGION}')
print(f'   Raw bucket:       s3://{RAW_BUCKET}/')
print(f'   Processed bucket: s3://{PROCESSED_BUCKET}/')
print(f'   Key ID ends in:   ...{AWS_ACCESS_KEY_ID[-4:]}  (masked)')

🔑 Credentials loaded from Colab Secrets
   Region:           eu-north-1
   Raw bucket:       s3://fraud-raw-chulu/
   Processed bucket: s3://fraud-processed-chulu/
   Key ID ends in:   ...ZAF3  (masked)


### 1.3 Generate synthetic dataset

| Column | Type | Description |
|---|---|---|
| `transaction_id` | string | Unique UUID per transaction |
| `user_id` | string | 5,000 unique customers |
| `timestamp` | datetime | Jan–Mar 2024 window |
| `amount` | float | Log-normal spend distribution |
| `merchant` | string | 25 real-world merchants |
| `merchant_category` | string | Retail, Travel, Food, etc. |
| `country` | string | ISO-2 country code |
| `card_present` | bool | In-person vs online |
| `is_fraud` | bool | Ground truth label |
| `fraud_reason` | string | Fraud type or NONE |

In [3]:
import numpy as np
import pandas as pd
import uuid
import random
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

N_TRANSACTIONS = 300_000
N_USERS        = 5_000
FRAUD_RATE     = 0.015
START_DATE     = datetime(2024, 1, 1)
END_DATE       = datetime(2024, 3, 31)

MERCHANTS = [
    ('Amazon',          'Online Retail'),  ('Walmart',       'Retail'),
    ('Target',          'Retail'),         ('Shell',         'Fuel'),
    ('BP',              'Fuel'),           ('Uber',          'Transport'),
    ('Lyft',            'Transport'),      ('Delta Airlines','Travel'),
    ('Marriott',        'Travel'),         ("McDonald's",    'Food'),
    ('Starbucks',       'Food'),           ('Netflix',       'Subscription'),
    ('Spotify',         'Subscription'),   ('Apple Store',   'Electronics'),
    ('Best Buy',        'Electronics'),    ('CVS Pharmacy',  'Healthcare'),
    ('Walgreens',       'Healthcare'),     ('Home Depot',    'Home Improvement'),
    ('Ikea',            'Home Improvement'),('Zara',         'Apparel'),
    ('H&M',             'Apparel'),        ('Steam',         'Gaming'),
    ('PlayStation Store','Gaming'),        ('Airbnb',        'Travel'),
    ('Booking.com',     'Travel'),
]

HOME_COUNTRIES    = ['US']*70 + ['UK']*10 + ['CA']*10 + ['AU']*5 + ['DE']*5
FOREIGN_COUNTRIES = ['NG','RU','CN','BR','PK','UA','RO','VN','KE','GH']

user_ids          = [f'USR{str(i).zfill(5)}' for i in range(N_USERS)]
user_home_country = {uid: random.choice(HOME_COUNTRIES) for uid in user_ids}
date_range_sec    = int((END_DATE - START_DATE).total_seconds())

print(f'⏳ Generating {N_TRANSACTIONS:,} transactions...')

rows = []
for _ in range(N_TRANSACTIONS):
    user     = random.choice(user_ids)
    merchant, category = random.choice(MERCHANTS)
    ts       = START_DATE + timedelta(seconds=random.randint(0, date_range_sec))
    amount   = round(min(np.random.lognormal(3.5, 1.2), 25000), 2)
    card_present = random.random() > 0.35
    country  = user_home_country[user]
    is_fraud = False
    fraud_reason = 'NONE'

    if random.random() < FRAUD_RATE:
        is_fraud = True
        fraud_type = random.choices(
            ['high_amount', 'foreign_transaction', 'card_not_present', 'rapid_succession'],
            weights=[30, 35, 20, 15]
        )[0]
        if fraud_type == 'high_amount':
            amount = round(random.uniform(5000, 25000), 2)
            fraud_reason = 'HIGH_AMOUNT'
        elif fraud_type == 'foreign_transaction':
            country = random.choice(FOREIGN_COUNTRIES)
            fraud_reason = 'FOREIGN_COUNTRY'
        elif fraud_type == 'card_not_present':
            card_present = False
            amount = round(random.uniform(1000, 8000), 2)
            fraud_reason = 'HIGH_VALUE_CNP'
        else:
            fraud_reason = 'RAPID_SUCCESSION'

    rows.append({
        'transaction_id':    str(uuid.uuid4()),
        'user_id':           user,
        'timestamp':         ts.strftime('%Y-%m-%d %H:%M:%S'),
        'amount':            amount,
        'merchant':          merchant,
        'merchant_category': category,
        'country':           country,
        'card_present':      card_present,
        'is_fraud':          is_fraud,
        'fraud_reason':      fraud_reason,
    })

df = pd.DataFrame(rows).sort_values('timestamp').reset_index(drop=True)

print(f'✅ Dataset generated')
print(f'   Rows:         {len(df):,}')
print(f'   Fraud count:  {df["is_fraud"].sum():,} ({df["is_fraud"].mean()*100:.2f}%)')
print(f'   Date range:   {df["timestamp"].min()} → {df["timestamp"].max()}')
print(f'   Amount range: ${df["amount"].min():.2f} – ${df["amount"].max():.2f}')
print('\n   Fraud breakdown:')
for reason, cnt in df[df['is_fraud']]['fraud_reason'].value_counts().items():
    print(f'     {reason:<22} {cnt:,}')

⏳ Generating 300,000 transactions...
✅ Dataset generated
   Rows:         300,000
   Fraud count:  4,482 (1.49%)
   Date range:   2024-01-01 00:00:11 → 2024-03-30 23:59:40
   Amount range: $0.16 – $24995.10

   Fraud breakdown:
     FOREIGN_COUNTRY        1,519
     HIGH_AMOUNT            1,365
     HIGH_VALUE_CNP         937
     RAPID_SUCCESSION       661


### 1.4 Preview & validate schema

In [4]:
print('=== Schema ===')
print(df.dtypes)
print('\n=== Null check ===')
print(df.isnull().sum())
print('\n=== Sample fraudulent transactions ===')
df[df['is_fraud']].head(5)

=== Schema ===
transaction_id        object
user_id               object
timestamp             object
amount               float64
merchant              object
merchant_category     object
country               object
card_present            bool
is_fraud                bool
fraud_reason          object
dtype: object

=== Null check ===
transaction_id       0
user_id              0
timestamp            0
amount               0
merchant             0
merchant_category    0
country              0
card_present         0
is_fraud             0
fraud_reason         0
dtype: int64

=== Sample fraudulent transactions ===


,transaction_id,user_id,timestamp,amount,merchant,merchant_category,country,card_present,is_fraud,fraud_reason
2,b788fa0b-ddd6-43a6-9a5d-5ea8098b06b1,USR01654,2024-01-01 00:00:35,58.11,Amazon,Online Retail,PK,False,True,FOREIGN_COUNTRY
9,8ecf5131-4f2f-4d8a-9d11-1a653f85d2ce,USR01578,2024-01-01 00:02:54,15909.51,Home Depot,Home Improvement,US,True,True,HIGH_AMOUNT
40,31019522-90e9-4833-9f98-480a6331e11d,USR01040,2024-01-01 00:16:14,12894.13,PlayStation Store,Gaming,US,True,True,HIGH_AMOUNT
82,01267645-bea9-479d-97b7-df5092cc0b50,USR00913,2024-01-01 00:33:07,8247.85,H&M,Apparel,UK,True,True,HIGH_AMOUNT
140,476bed9c-d9d3-4ec6-81af-5860a3b10141,USR00792,2024-01-01 01:00:11,7685.74,Airbnb,Travel,US,False,True,HIGH_VALUE_CNP


### 1.5 Save CSV locally

In [5]:
import os

OUTPUT_DIR = '/content/fraud_data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CSV_PATH = os.path.join(OUTPUT_DIR, 'transactions_raw.csv')

df.to_csv(CSV_PATH, index=False)
size_mb = os.path.getsize(CSV_PATH) / 1e6
print(f'✅ Saved: {CSV_PATH}  ({size_mb:.1f} MB)')

✅ Saved: /content/fraud_data/transactions_raw.csv  (32.8 MB)


### 1.6 Connect to AWS S3

In [6]:
import boto3
from botocore.exceptions import ClientError

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)
s3 = session.client('s3')

try:
    s3.list_buckets()
    print('✅ AWS connection successful')
except ClientError as e:
    print(f'❌ Connection failed: {e}')
    raise

✅ AWS connection successful


### 1.7 Create S3 buckets

In [7]:
def create_bucket(client, name, region):
    try:
        if region == 'us-east-1':
            client.create_bucket(Bucket=name)
        else:
            client.create_bucket(
                Bucket=name,
                CreateBucketConfiguration={'LocationConstraint': region}
            )
        print(f'   ✅ Created s3://{name}/')
    except ClientError as e:
        code = e.response['Error']['Code']
        if code in ('BucketAlreadyOwnedByYou', 'BucketAlreadyExists'):
            print(f'   ℹ️  Already exists: s3://{name}/')
        else:
            raise

print('Creating S3 buckets...')
create_bucket(s3, RAW_BUCKET, AWS_REGION)
create_bucket(s3, PROCESSED_BUCKET, AWS_REGION)

# 90-day auto-expiry on processed bucket (cost safety net)
s3.put_bucket_lifecycle_configuration(
    Bucket=PROCESSED_BUCKET,
    LifecycleConfiguration={
        'Rules': [{
            'ID': 'expire-processed-data',
            'Status': 'Enabled',
            'Filter': {'Prefix': ''},
            'Expiration': {'Days': 90}
        }]
    }
)
print('   ✅ 90-day lifecycle rule set on processed bucket')

Creating S3 buckets...
   ✅ Created s3://fraud-raw-chulu/
   ✅ Created s3://fraud-processed-chulu/
   ✅ 90-day lifecycle rule set on processed bucket


### 1.8 Upload CSV to S3

In [8]:
import sys

S3_RAW_KEY = 'raw/transactions_raw.csv'
file_size  = os.path.getsize(CSV_PATH)
uploaded   = [0]

def progress(chunk):
    uploaded[0] += chunk
    pct = uploaded[0] / file_size * 100
    bar = '█' * int(pct // 5) + '░' * (20 - int(pct // 5))
    sys.stdout.write(f'\r   [{bar}] {pct:.1f}%  ({uploaded[0]/1e6:.1f}/{file_size/1e6:.1f} MB)')
    sys.stdout.flush()

print(f'⏳ Uploading → s3://{RAW_BUCKET}/{S3_RAW_KEY}')
s3.upload_file(Filename=CSV_PATH, Bucket=RAW_BUCKET, Key=S3_RAW_KEY, Callback=progress)

verified_size = s3.head_object(Bucket=RAW_BUCKET, Key=S3_RAW_KEY)['ContentLength'] / 1e6
print(f'\n✅ Verified in S3: {verified_size:.1f} MB')

⏳ Uploading → s3://fraud-raw-chulu/raw/transactions_raw.csv
   [████████████████████] 100.0%  (32.8/32.8 MB)
✅ Verified in S3: 32.8 MB


### 1.9 Summary

In [9]:
print('=' * 60)
print('  PHASE 1 COMPLETE — save these for Phase 2')
print('=' * 60)
print(f'''
  AWS Region:         {AWS_REGION}
  Raw S3 path:        s3://{RAW_BUCKET}/raw/
  Processed S3 path:  s3://{PROCESSED_BUCKET}/processed/
  CSV S3 key:         {S3_RAW_KEY}

  Rows generated:     {len(df):,}
  Fraud records:      {df["is_fraud"].sum():,} ({df["is_fraud"].mean()*100:.2f}%)
  Unique users:       {df["user_id"].nunique():,}
  Countries present:  {sorted(df["country"].unique())}

  Next step → Phase 2: Glue Crawler + ETL Job
  Estimated cost:     ~$0.44 (one Glue job run)
''')
print('=' * 60)
print('  Cost so far: $0.00')
print('=' * 60)

  PHASE 1 COMPLETE — save these for Phase 2

  AWS Region:         eu-north-1
  Raw S3 path:        s3://fraud-raw-chulu/raw/
  Processed S3 path:  s3://fraud-processed-chulu/processed/
  CSV S3 key:         raw/transactions_raw.csv

  Rows generated:     300,000
  Fraud records:      4,482 (1.49%)
  Unique users:       5,000
  Countries present:  ['AU', 'BR', 'CA', 'CN', 'DE', 'GH', 'KE', 'NG', 'PK', 'RO', 'RU', 'UA', 'UK', 'US', 'VN']

  Next step → Phase 2: Glue Crawler + ETL Job
  Estimated cost:     ~$0.44 (one Glue job run)

  Cost so far: $0.00


---

## Phase 5 — Interactive Dashboard

> **Prerequisite:** Phases 2–4 must be complete in the AWS Console (Glue Crawler, Glue ETL job, Athena queries). The `s3` client and `PROCESSED_BUCKET` from Phase 1 are reused here — run this in the same Colab session or re-run Cell 1.2 first.

### 5.1 Install dashboard dependencies

In [ ]:
!pip install plotly pyarrow --quiet
print('✅ Dashboard dependencies installed')

### 5.2 Download processed Parquet data from S3

In [ ]:
import os
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

LOCAL_DIR = '/content/processed'
os.makedirs(LOCAL_DIR, exist_ok=True)

paginator = s3.get_paginator('list_objects_v2')
pages = paginator.paginate(Bucket=PROCESSED_BUCKET, Prefix='processed/')

parquet_keys = []
for page in pages:
    for obj in page.get('Contents', []):
        if obj['Key'].endswith('.parquet'):
            parquet_keys.append(obj['Key'])

if not parquet_keys:
    raise FileNotFoundError(
        f'No Parquet files found in s3://{PROCESSED_BUCKET}/processed/ '
        'Run the Glue ETL job (Phase 3) first.'
    )

print(f'Found {len(parquet_keys)} partition files — downloading...')
tables = []
for i, key in enumerate(parquet_keys):
    local_path = os.path.join(LOCAL_DIR, key.replace('/', '_'))
    s3.download_file(PROCESSED_BUCKET, key, local_path)
    tables.append(pq.read_table(local_path))
    if (i + 1) % 5 == 0 or (i + 1) == len(parquet_keys):
        print(f'  {i+1}/{len(parquet_keys)} files downloaded')

df_processed = pa.concat_tables(tables).to_pandas()
df_processed['amount']        = df_processed['amount'].astype(float)
df_processed['is_fraud']      = df_processed['is_fraud'].astype(bool)
df_processed['is_high_value'] = df_processed['is_high_value'].astype(bool)
df_processed['timestamp']     = pd.to_datetime(df_processed['timestamp'])
df_processed['txn_date']      = pd.to_datetime(df_processed['txn_date'])

print(f'\n✅ {len(df_processed):,} rows loaded')
print(f'   Fraud records: {df_processed["is_fraud"].sum():,} ({df_processed["is_fraud"].mean()*100:.2f}%)')


### 5.3 Aggregate metrics for each chart

In [ ]:
# KPI metrics
total_txns      = len(df_processed)
total_fraud     = int(df_processed['is_fraud'].sum())
fraud_rate      = df_processed['is_fraud'].mean() * 100
total_fraud_amt = df_processed[df_processed['is_fraud']]['amount'].sum()

# Fraud by reason
fraud_by_reason = (
    df_processed[df_processed['is_fraud']]
    .groupby('fraud_reason')
    .agg(count=('transaction_id','count'), total_amount=('amount','sum'))
    .reset_index()
    .sort_values('count', ascending=False)
)

# Fraud by country
fraud_by_country = (
    df_processed[df_processed['is_fraud']]
    .groupby('country')
    .agg(count=('transaction_id','count'), total_amount=('amount','sum'))
    .reset_index()
    .sort_values('total_amount', ascending=True)
)

# Daily fraud trend
fraud_daily = (
    df_processed[df_processed['is_fraud']]
    .groupby('txn_date')
    .agg(fraud_count=('transaction_id','count'), fraud_amount=('amount','sum'))
    .reset_index()
)

# High-risk leaderboard
user_stats = df_processed.groupby('user_id').agg(
    total_transactions=('transaction_id','count'),
    total_spend=('amount','sum'),
    confirmed_fraud=('is_fraud','sum'),
    max_amount=('amount','max'),
).reset_index()

fraud_breakdown = (
    df_processed[df_processed['is_fraud']]
    .groupby(['user_id','fraud_reason'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
for col in ['HIGH_AMOUNT','FOREIGN_COUNTRY','HIGH_VALUE_CNP','RAPID_SUCCESSION']:
    if col not in fraud_breakdown.columns:
        fraud_breakdown[col] = 0

fraud_breakdown['risk_score'] = (
    fraud_breakdown['HIGH_AMOUNT']      * 3 +
    fraud_breakdown['FOREIGN_COUNTRY']  * 4 +
    fraud_breakdown['HIGH_VALUE_CNP']   * 3 +
    fraud_breakdown['RAPID_SUCCESSION'] * 2
)

leaderboard = (
    user_stats
    .merge(fraud_breakdown[['user_id','risk_score','HIGH_AMOUNT',
                             'FOREIGN_COUNTRY','HIGH_VALUE_CNP','RAPID_SUCCESSION']],
           on='user_id', how='inner')
    .query('confirmed_fraud > 0')
    .sort_values('risk_score', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
leaderboard.index += 1

print('✅ Aggregations ready')
print(f'   {total_fraud:,} fraud flags | {fraud_rate:.2f}% rate | ${total_fraud_amt:,.0f} total amount')


### 5.4 Build and display dashboard

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

BG       = '#0F1117'
CARD     = '#1A1D27'
TEXT     = '#E8EAF0'
GRID     = '#2A2D3A'
BLUE     = '#1D3557'
RED      = '#E63946'
REASONS  = {
    'HIGH_AMOUNT':      '#E63946',
    'FOREIGN_COUNTRY':  '#F4A261',
    'HIGH_VALUE_CNP':   '#E9C46A',
    'RAPID_SUCCESSION': '#A8DADC',
}

fig = make_subplots(
    rows=3, cols=3,
    specs=[
        [{"type":"indicator"}, {"type":"indicator"}, {"type":"indicator"}],
        [{"type":"pie"},       {"type":"bar","colspan":2}, None],
        [{"type":"scatter","colspan":2}, None, {"type":"table"}],
    ],
    subplot_titles=("","","",
                    "Fraud by Type", "Fraud Amount by Country",
                    "Daily Fraud Trend", "High-Risk Leaderboard"),
    vertical_spacing=0.10,
    horizontal_spacing=0.06,
    row_heights=[0.18, 0.38, 0.44]
)

# KPI cards
for col, val, fmt, color, label, prefix, suffix in [
    (1, total_fraud,     ",",    "#E63946", "Total Fraud Flags",   None, None),
    (2, fraud_rate,      ".2f",  "#F4A261", "Fraud Rate",          None, "%"),
    (3, total_fraud_amt, ",.0f", "#E9C46A", "Total Fraud Amount",  "$",  None),
]:
    num = dict(font=dict(size=38, color=color, family="Arial"), valueformat=fmt)
    if prefix: num['prefix'] = prefix
    if suffix: num['suffix'] = suffix
    fig.add_trace(go.Indicator(
        mode="number", value=val, number=num,
        title=dict(text=label, font=dict(size=13, color="#9099B0", family="Arial")),
    ), row=1, col=col)

# Donut: fraud by type
fig.add_trace(go.Pie(
    labels=fraud_by_reason['fraud_reason'],
    values=fraud_by_reason['count'],
    hole=0.45,
    marker=dict(colors=[REASONS.get(r,'#888') for r in fraud_by_reason['fraud_reason']],
                line=dict(color=BG, width=2)),
    textinfo='label+percent',
    textfont=dict(size=10, color=TEXT, family="Arial"),
    showlegend=False,
), row=2, col=1)

# Bar: fraud by country
fig.add_trace(go.Bar(
    x=fraud_by_country['total_amount'],
    y=fraud_by_country['country'],
    orientation='h',
    marker=dict(color=fraud_by_country['total_amount'],
                colorscale=[[0,'#457B9D'],[0.5,RED],[1,'#FF6B6B']],
                showscale=False, line=dict(width=0)),
    text=[f'${v:,.0f}' for v in fraud_by_country['total_amount']],
    textposition='outside',
    textfont=dict(size=10, color=TEXT, family="Arial"),
    hovertemplate='<b>%{y}</b><br>$%{x:,.0f}<extra></extra>',
), row=2, col=2)

# Area line: daily trend
fig.add_trace(go.Scatter(
    x=fraud_daily['txn_date'], y=fraud_daily['fraud_count'],
    mode='lines+markers',
    line=dict(color=RED, width=2.5, shape='spline'),
    marker=dict(size=4, color=RED),
    fill='tozeroy', fillcolor='rgba(230,57,70,0.12)',
    hovertemplate='<b>%{x|%b %d}</b><br>Flags: %{y}<extra></extra>',
    showlegend=False,
), row=3, col=1)

# Table: leaderboard
top_reason = leaderboard[
    ['HIGH_AMOUNT','FOREIGN_COUNTRY','HIGH_VALUE_CNP','RAPID_SUCCESSION']
].idxmax(axis=1)

fig.add_trace(go.Table(
    columnwidth=[110,70,70,90,110],
    header=dict(
        values=['<b>User ID</b>','<b>Risk Score</b>','<b>Fraud Count</b>',
                '<b>Max Amount</b>','<b>Top Reason</b>'],
        fill_color=BLUE, font=dict(color=TEXT, size=11, family="Arial"),
        align='center', line_color=GRID, height=28,
    ),
    cells=dict(
        values=[
            leaderboard['user_id'], leaderboard['risk_score'],
            leaderboard['confirmed_fraud'].astype(int),
            ['$'+f'{v:,.0f}' for v in leaderboard['max_amount']],
            top_reason,
        ],
        fill_color=[[CARD if i%2==0 else '#20232F' for i in range(len(leaderboard))]],
        font=dict(color=TEXT, size=10, family="Arial"),
        align=['left','center','center','right','center'],
        line_color=GRID, height=24,
    ),
), row=3, col=3)

ax = dict(gridcolor=GRID, gridwidth=0.5, linecolor=GRID,
          tickfont=dict(color='#9099B0', size=10, family="Arial"), zeroline=False)

fig.update_layout(
    title=dict(text='Batch Fraud Analytics Dashboard',
               font=dict(size=22, color=TEXT, family="Arial"), x=0.02, y=0.99),
    paper_bgcolor=BG, plot_bgcolor=CARD,
    font=dict(color=TEXT, family="Arial"),
    height=880, margin=dict(t=60, b=40, l=40, r=40),
    annotations=[a for a in fig.layout.annotations] + [dict(
        text=f'{total_txns:,} transactions | {total_fraud:,} fraud flags ({fraud_rate:.2f}%) | Jan–Mar 2024',
        xref='paper', yref='paper', x=0.02, y=-0.02,
        font=dict(size=10, color='#6B7280'), showarrow=False, align='left'
    )]
)
fig.update_xaxes(**ax)
fig.update_yaxes(**ax)
for ann in fig.layout.annotations:
    ann.font = dict(size=13, color='#C5CAD9', family='Arial')

fig.show()
print('✅ Dashboard rendered')


### 5.5 Export to HTML and publish

After running the cell below, download `dashboard.html` from the Colab Files panel and add it to your repo at `docs/dashboard.html`. Enable GitHub Pages (Settings → Pages → Branch: main → Folder: /docs) to get a permanent public URL:

`https://YOUR_USERNAME.github.io/fraud-pipeline/dashboard.html`

In [ ]:
DASHBOARD_PATH = '/content/dashboard.html'

fig.write_html(
    DASHBOARD_PATH,
    include_plotlyjs=True,
    full_html=True,
    config={
        'displayModeBar': True,
        'displaylogo': False,
        'modeBarButtonsToRemove': ['sendDataToCloud'],
        'toImageButtonOptions': {
            'format': 'png', 'filename': 'fraud_dashboard',
            'height': 900, 'width': 1600, 'scale': 2
        }
    }
)

size_mb = os.path.getsize(DASHBOARD_PATH) / 1e6
print(f'✅ Exported dashboard.html  ({size_mb:.1f} MB)')

s3.upload_file(
    Filename=DASHBOARD_PATH,
    Bucket=PROCESSED_BUCKET,
    Key='dashboard/dashboard.html',
    ExtraArgs={'ContentType': 'text/html'}
)
print(f'✅ Backed up to s3://{PROCESSED_BUCKET}/dashboard/dashboard.html')
print()
print('Next: download dashboard.html from the Colab Files panel')
print('then push to docs/dashboard.html and enable GitHub Pages.')
